# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abood-arc/Flyrank-ml-project/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [57]:
%pip -q install duckdb huggingface_hub

import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT_MAR    = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DIM_CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

print("""One row = one content item, on one report_date, for one client.
Table: fact_content_daily_performance (grain: report_date + client_hash_id + content_hash_id),
joined against dim_content and dim_clients for context and enrichment -- never for a different
grain, they're many-rows-to-one joins that add columns, not rows.

Time window: month = 2026-03. A mid-panel month, not the _sample table and not the final
month """)

One row = one content item, on one report_date, for one client.
Table: fact_content_daily_performance (grain: report_date + client_hash_id + content_hash_id),
joined against dim_content and dim_clients for context and enrichment -- never for a different
grain, they're many-rows-to-one joins that add columns, not rows.

Time window: month = 2026-03. A mid-panel month, not the _sample table and not the final
month 


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [41]:
print("""FEATURE -- gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position
  gated by gsc_data_available IS TRUE.

FEATURE -- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions,
  ga4_total_engagement_sec, scroll_events, sessions_organic/direct/referral/social/paid/ai,
  ai_chatgpt/perplexity/gemini/copilot/claude/meta/other
  gated by ga4_data_available IS TRUE -- and thin: only ~4% of March rows qualify (section 3).

FEATURE -- content_type, word_count, char_count, search_volume, competition, competition_level,
  cpc, main_intent, keyword_char_count, keyword_token_count, url_char_count
  static dim_content metadata, joined on content_hash_id.

LABEL / PROXY -- a decline signal I build myself from consecutive report_date rows for the
  same content_hash_id -- not a column that ships in this table. (Section 3's trap uses a
  within-month first-half-vs-second-half stand-in for this; the real capstone label is the
  client-relative one already locked in w02_ml_task_framing.ipynb, not this.)

CONTEXT -- report_date, client_hash_id, content_hash_id (grain key)
  client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available (availability flags,
  repeat per row, never sum)
  dim_clients: is_active, has_gsc_access, has_ga4_access, access_profile, gsc_data_start,
  ga4_data_start -- population eligibility and history-depth checks, not a page-level signal.

EXCLUDED
  last_optimized_date, optimization_eligible_date -- downstream of a product action (an editor
    already decided to refresh this page), so it's a proxy for a decision already made, not an
    observation. Also a snapshot field, see next line.
  is_deleted, is_published -- dim_content is a single export-time snapshot, not a per-report_date
    history, so these can describe a state that didn't exist yet as of an earlier report_date.
  provider_used, model_used -- undocumented anywhere I've read. Not using a column I can't explain.
  backlinks, category_count -- same snapshot-timing concern as above, and also undocumented.""")

FEATURE -- gsc_impressions, gsc_clicks, gsc_sum_position, gsc_avg_position
  gated by gsc_data_available IS TRUE.

FEATURE -- ga4_pageviews, ga4_sessions, ga4_users, ga4_engaged_sessions,
  ga4_total_engagement_sec, scroll_events, sessions_organic/direct/referral/social/paid/ai,
  ai_chatgpt/perplexity/gemini/copilot/claude/meta/other
  gated by ga4_data_available IS TRUE -- and thin: only ~4% of March rows qualify (section 3).

FEATURE -- content_type, word_count, char_count, search_volume, competition, competition_level,
  cpc, main_intent, keyword_char_count, keyword_token_count, url_char_count
  static dim_content metadata, joined on content_hash_id.

LABEL / PROXY -- a decline signal I build myself from consecutive report_date rows for the
  same content_hash_id -- not a column that ships in this table. (Section 3's trap uses a
  within-month first-half-vs-second-half stand-in for this; the real capstone label is the
  client-relative one already locked in w02_ml_task_framing.ipyn

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### Fact 1 — grain (one row really is one content item × one report_date × one client)

In [42]:
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
    FROM {FACT_MAR}
    GROUP BY report_date, client_hash_id, content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()

print(f'duplicate grain rows found: {len(grain_check)}')
grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate grain rows found: 0


,report_date,client_hash_id,content_hash_id,n


### Fact 2 — row count and date span

In [43]:
counts = con.sql(f"""
    SELECT COUNT(*)                       AS row_count,
           COUNT(DISTINCT client_hash_id)  AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date)                AS min_date,
           MAX(report_date)                AS max_date
    FROM {FACT_MAR}
""").df()

counts


,row_count,n_clients,n_content,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


### Fact 3 — availability, filtered with `IS TRUE` (not `= TRUE`, which mishandles NULL)

In [44]:
availability = con.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)     AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)     AS ga4_available_rows
    FROM {FACT_MAR}
""").df()

availability['gsc_available_pct'] = availability['gsc_available_rows'] / availability['total_rows']
availability['ga4_available_pct'] = availability['ga4_available_rows'] / availability['total_rows']
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,gsc_available_pct,ga4_available_pct
0,9841378,3611061.0,413966.0,0.366926,0.042064


### Five features, max — each with a one-line "knowable when"

Raw GSC/GA4 columns are zero-filled when tracking wasn't available, so each metric is masked
with its own availability flag below — unavailable becomes real `NULL`, not a fake 0 (the
"two kinds of nothing" lesson).

In [45]:
features = con.sql(f"""
    SELECT
        f.report_date,
        f.client_hash_id,
        f.content_hash_id,
        CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_impressions END      AS gsc_impressions,
        CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_clicks END           AS gsc_clicks,
        CASE WHEN f.gsc_data_available IS TRUE THEN f.gsc_avg_position END     AS gsc_avg_position,
        CASE WHEN f.ga4_data_available IS TRUE THEN f.ga4_engaged_sessions END AS ga4_engaged_sessions,
        c.content_type
    FROM {FACT_MAR} f
    LEFT JOIN {DIM_CONTENT} c ON f.content_hash_id = c.content_hash_id
""").df()

print(f'{len(features):,} rows')
print(features.isna().mean().round(3))

print("""
gsc_impressions       knowable -- finalized GSC count for a report_date that has already
                       completed and synced through the nightly pipeline.
gsc_clicks             knowable -- same, closed-day aggregate.
gsc_avg_position       knowable -- same.
content_type           knowable -- set at publish time, before any report_date in this slice.
ga4_engaged_sessions   knowable -- same closed-day logic as GSC, but only where
                       ga4_data_available IS TRUE -- true for just ~4% of March rows.
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

9,841,378 rows
report_date             0.000
client_hash_id          0.000
content_hash_id         0.000
gsc_impressions         0.633
gsc_clicks              0.633
gsc_avg_position        0.633
ga4_engaged_sessions    0.958
content_type            0.000
dtype: float64

gsc_impressions       knowable -- finalized GSC count for a report_date that has already
                       completed and synced through the nightly pipeline.
gsc_clicks             knowable -- same, closed-day aggregate.
gsc_avg_position       knowable -- same.
content_type           knowable -- set at publish time, before any report_date in this slice.
ga4_engaged_sessions   knowable -- same closed-day logic as GSC, but only where
                       ga4_data_available IS TRUE -- true for just ~4% of March rows.



### The trap — a deliberate leak, then removed

Demo label only, scoped to this exercise — **not** the capstone label (that's the
client-relative one already locked in `w02_ml_task_framing.ipynb`). Splits March itself in
half so there's a feature window and a label window to violate on purpose, mirroring
notebook 02's `trend_pct` / `trend_direction` leak on real warehouse data.

In [46]:
half_split = con.sql(f"""
    WITH agg AS (
        SELECT
            content_hash_id,
            SUM(CASE WHEN report_date <  DATE '2026-03-16' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS imp_first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' AND gsc_data_available IS TRUE THEN gsc_impressions ELSE 0 END) AS imp_second_half,
            SUM(CASE WHEN report_date <  DATE '2026-03-16' AND gsc_data_available IS TRUE THEN gsc_clicks ELSE 0 END)      AS clk_first_half,
            AVG(CASE WHEN report_date <  DATE '2026-03-16' AND gsc_data_available IS TRUE THEN gsc_avg_position END)      AS pos_first_half,
            SUM(CASE WHEN report_date <  DATE '2026-03-16' AND ga4_data_available IS TRUE THEN ga4_engaged_sessions ELSE 0 END) AS eng_first_half
        FROM {FACT_MAR}
        GROUP BY content_hash_id
        HAVING imp_first_half >= 20
    )
    SELECT a.*, c.content_type
    FROM agg a
    LEFT JOIN {DIM_CONTENT} c ON a.content_hash_id = c.content_hash_id
""").df()

half_split['pct_change']   = (half_split['imp_second_half'] - half_split['imp_first_half']) / half_split['imp_first_half']
half_split['is_declining'] = (half_split['pct_change'] < -0.2).astype(int)
print(f"{len(half_split):,} content items | declining rate: {half_split['is_declining'].mean():.3f}")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

109,592 content items | declining rate: 0.291


In [54]:
from sklearn.tree import DecisionTreeClassifier, export_text
import numpy as np

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

honest_features = ['imp_first_half', 'clk_first_half', 'pos_first_half', 'eng_first_half']
X = half_split[honest_features].fillna(0)
y = half_split['is_declining']

tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X, y)
honest_p50 = precision_at_k(tree.predict_proba(X)[:, 1], y, 50)
print(f'Honest Precision@50: {honest_p50:.3f}')
print(export_text(tree, feature_names=honest_features))


Honest Precision@50: 0.460
|--- clk_first_half <= 3.50
|   |--- pos_first_half <= 11.05
|   |   |--- class: 1
|   |--- pos_first_half >  11.05
|   |   |--- class: 0
|--- clk_first_half >  3.50
|   |--- pos_first_half <= 22.11
|   |   |--- class: 0
|   |--- pos_first_half >  22.11
|   |   |--- class: 1



In [53]:
leaky_features = honest_features + ['pct_change']
X_leaky = half_split[leaky_features].fillna(0)

leaky_tree = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42).fit(X_leaky, y)
leaky_p50 = precision_at_k(leaky_tree.predict_proba(X_leaky)[:, 1], y, 50)
print(f"'Leaky' Precision@50: {leaky_p50:.3f}  <- pct_change IS the label in disguise")
print(export_text(leaky_tree, feature_names=leaky_features))


'Leaky' Precision@50: 1.000  <- pct_change IS the label in disguise
|--- pct_change <= -0.20
|   |--- class: 1
|--- pct_change >  -0.20
|   |--- class: 0



In [52]:
# pct_change dropped -- it's the label's own input, not a feature.
print(f'Final, honest Precision@50: {honest_p50:.3f}')


Final, honest Precision@50: 0.460


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [47]:
print("""GA4 coverage in this slice is thin -- 4.2% of March's rows have ga4_data_available =
TRUE. Ninety-six percent of the panel is a page Search Console watched but nobody's
analytics tool did. That's not "no engagement," it's "we weren't looking" -- the exact
mix-up the masking in section 3 exists to prevent.

Practically: any GA4-based feature is only real evidence for a sliver of pages. Most of what
this slice can honestly tell me comes from search visibility alone.

One more, smaller: dim_content is a single export-time snapshot, not a day-by-day history --
which is exactly why last_optimized_date and friends are excluded, not featured.""")


GA4 coverage in this slice is thin -- 4.2% of March's rows have ga4_data_available =
TRUE. Ninety-six percent of the panel is a page Search Console watched but nobody's
analytics tool did. That's not "no engagement," it's "we weren't looking" -- the exact
mix-up the masking in section 3 exists to prevent.

Practically: any GA4-based feature is only real evidence for a sliver of pages. Most of what
this slice can honestly tell me comes from search visibility alone.

One more, smaller: dim_content is a single export-time snapshot, not a day-by-day history --
which is exactly why last_optimized_date and friends are excluded, not featured.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.